# Automated FOG Labeling Pipeline — All Participants

This notebook processes all participants and tasks automatically from OneDrive.

**For each (participant, task) pair it:**
1. Finds the matching H5 file and EAF file
2. Extracts FOG + task labels from the EAF
3. Labels all 5 IMU devices using LSL synchronization
4. Resamples to 100 Hz
5. Saves one CSV per device: `IMU_devX_FOGXXX_taskname_labeled_100Hz.csv`

**Output folder:** set in parameters below

## 1. Imports

In [ ]:
import h5py
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from scipy.interpolate import interp1d
from pathlib import Path
import re
import os

## 2. Parameters

In [ ]:
# OneDrive base paths
ONEDRIVE = Path.home() / 'Library/CloudStorage/OneDrive-Bibliothèquespartagées-epfl.ch'

H5_BASE  = ONEDRIVE / 'Daniel Leal - Group_FOG (1)'
EAF_BASE = ONEDRIVE / 'Daniel Leal - Group_FOG'

# Output folder (local — CSVs are small)
OUTPUT_DIR = Path.home() / 'fog_labeled_data'
OUTPUT_DIR.mkdir(exist_ok=True)

# IMU devices to process
DEVICES     = ['dev2', 'dev3', 'dev5', 'dev6', 'dev7']
SIGNAL_COLS = ['acc_x', 'acc_y', 'acc_z', 'quat_w', 'quat_x', 'quat_y', 'quat_z']
TARGET_FS   = 100  # Hz — above native ~81 Hz, standard in IMU gait literature
CAMERA      = 'camera_1'

# EAF task keyword → H5 folder name
# Keys are lowercased substrings found in EAF filenames
TASK_MAP = {
    'circuit8':       '8 Shape Circuit_1',
    'obstacles2':     'Obstacles_2',
    'obstacles1':     'Obstacles_1',
    'obstacles':      'Obstacles_1',   # fallback (no number)
    'narrowcorridor2':'Narrow Corridor_2',
    'narrowcorridor1':'Narrow Corridor_1',
    'narrowcorridor': 'Narrow Corridor_1',  # fallback
    'stop':           'Stop Task_1',
}

print(f'H5 base  : {H5_BASE}')
print(f'EAF base : {EAF_BASE}')
print(f'Output   : {OUTPUT_DIR}')

## 3. Helper functions

In [ ]:
def eaf_task_key(eaf_path):
    """Extract task keyword from EAF filename, lowercased and stripped.
    e.g. 'task_narrowCorridor_FOG002.eaf' -> 'narrowcorridor'
         'task_obstacles2_FOG009.eaf'     -> 'obstacles2'
    """
    stem = Path(eaf_path).stem.lower()  # e.g. 'task_narrowcorridor_fog002'
    # Remove 'task_' prefix and '_fogXXX' suffix
    stem = re.sub(r'^task_', '', stem)
    stem = re.sub(r'_fog\d+$', '', stem)
    # Remove underscores and spaces for matching
    return stem.replace('_', '').replace(' ', '')


def match_task_folder(eaf_path):
    """Return the H5 task folder name for a given EAF file."""
    key = eaf_task_key(eaf_path)
    # Try longest match first to avoid 'obstacles' matching before 'obstacles2'
    for k in sorted(TASK_MAP.keys(), key=len, reverse=True):
        if key.startswith(k) or key == k:
            return TASK_MAP[k]
    return None


def parse_eaf(path):
    """Parse ELAN .eaf file. Returns fog_events and task_events."""
    tree = ET.parse(path)
    root = tree.getroot()
    time_slots = {
        ts.attrib['TIME_SLOT_ID']: int(ts.attrib['TIME_VALUE'])
        for ts in root.find('TIME_ORDER').findall('TIME_SLOT')
    }
    fog_events, task_events = [], []
    for tier in root.findall('TIER'):
        tier_id = tier.attrib.get('TIER_ID', '')
        for ann in tier.findall('.//ALIGNABLE_ANNOTATION'):
            t1  = time_slots[ann.attrib['TIME_SLOT_REF1']]
            t2  = time_slots[ann.attrib['TIME_SLOT_REF2']]
            val = ann.find('ANNOTATION_VALUE').text
            if tier_id == 'FOG':
                fog_events.append((t1, t2))
            elif tier_id == 'Task':
                task_events.append((t1, t2, val))
    return fog_events, task_events


def resample(ts_orig, data_orig, labels_orig, task_labels_orig, target_fs=100):
    """
    Resample IMU data to target_fs using vectorized operations.
    - Continuous signals: linear interpolation (scipy interp1d)
    - FOG label: conservative — 1 if ANY original sample in window was FOG
    - Task label: majority vote using np.searchsorted (fully vectorized, no Python loop)
    """
    t_new = np.arange(ts_orig[0], ts_orig[-1], 1.0 / target_fs)
    half  = 0.5 / target_fs

    # --- Continuous signals: linear interpolation ---
    data_new = np.zeros((len(t_new), data_orig.shape[1]))
    for ch in range(data_orig.shape[1]):
        f = interp1d(ts_orig, data_orig[:, ch], kind='linear',
                     bounds_error=False, fill_value='extrapolate')
        data_new[:, ch] = f(t_new)

    # --- Labels: vectorized using searchsorted ---
    # For each new sample t, find original indices in [t-half, t+half)
    idx_lo = np.searchsorted(ts_orig, t_new - half, side='left')
    idx_hi = np.searchsorted(ts_orig, t_new + half, side='right')

    fog_new  = np.zeros(len(t_new), dtype=int)
    task_new = np.zeros(len(t_new), dtype=int)

    # Only process windows that have at least one original sample
    has_samples = idx_hi > idx_lo

    # FOG: max in window (conservative — 1 if any FOG)
    for i in np.where(has_samples)[0]:
        fog_new[i] = labels_orig[idx_lo[i]:idx_hi[i]].max()

    # Task: mode in window — use np.bincount for speed
    n_tasks = int(task_labels_orig.max()) + 1
    for i in np.where(has_samples)[0]:
        window_tasks = task_labels_orig[idx_lo[i]:idx_hi[i]]
        task_new[i]  = np.bincount(window_tasks, minlength=n_tasks).argmax()

    return t_new, data_new, fog_new, task_new


# Quick test of task matching
test_cases = [
    'task_circuit8_FOG002.eaf',
    'task_narrowCorridor_FOG002.eaf',
    'task_narrowcorridor_FOG13.eaf',
    'task_narrowcorridor1_FOG011.eaf',
    'task_narrowcorridor2_FOG011.eaf',
    'task_obstacles_FOG002.eaf',
    'task_obstacles1_FOG009.eaf',
    'task_obstacles2_FOG009.eaf',
    'task_stop_FOG002.eaf',
]
print('Task matching test:')
for t in test_cases:
    print(f'  {t:45s} -> {match_task_folder(t)}')

## 4. Discover all (participant, EAF, H5) triplets

In [ ]:
# Find all EAF files grouped by participant
eaf_files = list(EAF_BASE.rglob('*.eaf'))

jobs = []  # list of (participant_id, eaf_path, h5_path, task_folder)
skipped = []

for eaf_path in sorted(eaf_files):
    # Extract participant ID from path
    parts = eaf_path.parts
    fog_id = None
    for p in parts:
        if re.match(r'FOG\d+', p):
            fog_id = p
            break
    if not fog_id:
        skipped.append((str(eaf_path), 'no participant ID found'))
        continue

    # Match to H5 task folder
    task_folder = match_task_folder(eaf_path.name)
    if not task_folder:
        skipped.append((eaf_path.name, 'no task folder match'))
        continue

    # Build H5 path
    h5_path = H5_BASE / fog_id / 'MAPP' / task_folder / 'recording_data.h5'
    if not h5_path.exists():
        skipped.append((eaf_path.name, f'H5 not found: {h5_path}'))
        continue

    jobs.append((fog_id, eaf_path, h5_path, task_folder))

print(f'Jobs ready: {len(jobs)}')
for fog_id, eaf, h5, task in jobs:
    print(f'  {fog_id} | {task:25s} | EAF: {eaf.name}')

if skipped:
    print(f'\nSkipped ({len(skipped)}):')
    for name, reason in skipped:
        print(f'  {name}: {reason}')

## 5. Run the full pipeline

In [ ]:
results = []  # summary log

for job_idx, (fog_id, eaf_path, h5_path, task_folder) in enumerate(jobs):
    task_name = task_folder.replace(' ', '_').replace('/', '_')
    print(f'\n[{job_idx+1}/{len(jobs)}] {fog_id} — {task_folder}')

    try:
        # --- Parse EAF ---
        fog_events, task_events = parse_eaf(eaf_path)
        task_labels_list = sorted(set(label for _, _, label in task_events))
        task_label_map   = {label: i+1 for i, label in enumerate(task_labels_list)}

        # --- Load camera timestamps from H5 ---
        with h5py.File(h5_path, 'r') as f:
            cam_raw = f[f'Cameras/{CAMERA}/frames'][:]
            cam_ts  = cam_raw['timestamp']
            t0_lsl  = cam_ts[0]

            # Convert EAF times to LSL
            fog_lsl  = [(t0_lsl + s/1000, t0_lsl + e/1000) for s, e in fog_events]
            task_lsl = [(t0_lsl + s/1000, t0_lsl + e/1000, lbl) for s, e, lbl in task_events]

            # --- Process each device ---
            for dev in DEVICES:
                ts_path   = f'WIMU/WIMU_MK3/timestamps_{dev}'
                data_path = f'WIMU/WIMU_MK3/data_{dev}'

                if ts_path not in f or data_path not in f:
                    print(f'  [{dev}] not found in H5, skipping')
                    continue

                imu_ts   = f[ts_path][:]
                imu_data = f[data_path][:]

                # Generate labels
                fog_labels  = np.zeros(len(imu_ts), dtype=int)
                task_labels = np.zeros(len(imu_ts), dtype=int)

                for t_start, t_end in fog_lsl:
                    fog_labels[(imu_ts >= t_start) & (imu_ts <= t_end)] = 1

                for t_start, t_end, lbl in task_lsl:
                    code = task_label_map.get(lbl, 0)
                    task_labels[(imu_ts >= t_start) & (imu_ts <= t_end)] = code

                # Resample to 100 Hz
                t_new, data_new, fog_new, task_new = resample(
                    imu_ts, imu_data, fog_labels, task_labels, TARGET_FS
                )

                # Build DataFrame
                col_names = ['timestamp_imu'] + SIGNAL_COLS
                df = pd.DataFrame(data_new, columns=col_names)
                df.insert(0, 'timestamp_lsl', t_new)
                df['fog_label']  = fog_new
                df['task_label'] = task_new
                df['subject_id'] = fog_id
                df['task_name']  = task_name

                # Save
                out_file = OUTPUT_DIR / f'IMU_{dev}_{fog_id}_{task_name}_labeled_100Hz.csv'
                df.to_csv(out_file, index=False)

                fog_pct = 100 * fog_new.mean()
                print(f'  [{dev}] {len(df)} samples | FOG: {fog_new.sum()} ({fog_pct:.1f}%) → {out_file.name}')

                results.append({
                    'subject': fog_id, 'task': task_name, 'device': dev,
                    'n_samples': len(df), 'fog_samples': int(fog_new.sum()),
                    'fog_pct': round(fog_pct, 2), 'output': out_file.name
                })

    except Exception as e:
        print(f'  ERROR: {e}')
        results.append({'subject': fog_id, 'task': task_name, 'device': 'ALL', 'error': str(e)})

print(f'\n✓ Pipeline complete — {len(results)} files generated')

## 6. Summary

In [ ]:
df_summary = pd.DataFrame(results)
print('=== Summary ===')
print(df_summary[['subject', 'task', 'device', 'n_samples', 'fog_samples', 'fog_pct']].to_string(index=False))

# Save summary
df_summary.to_csv(OUTPUT_DIR / 'pipeline_summary.csv', index=False)
print(f'\nSummary saved: {OUTPUT_DIR}/pipeline_summary.csv')

# FOG stats per participant
print('\n=== FOG % per participant (dev2 only) ===')
dev2 = df_summary[df_summary['device'] == 'dev2']
print(dev2.groupby('subject')[['fog_samples', 'n_samples']]
        .sum()
        .assign(fog_pct=lambda x: (100*x['fog_samples']/x['n_samples']).round(2))
        .to_string())